# Historical notebook — source archive

Original: `Copy_of_RAG_with_deepseek_Ved (4).ipynb`. Credentials and cloud endpoint removed; outputs cleared. This is provenance, **not a supported Run All workflow**. See the four notebooks above this archive for the maintained pipeline. Historical errors, global state, and incompatible APIs may remain. Cloud collection-reset cells have been disabled.


In [ ]:
import sqlite3
import os
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch

# Database setup
DB_PATH = "mllm_experiments.db"

def create_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS experiment_results_test2 (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            image_name TEXT,
            question TEXT,
            context TEXT,
            object TEXT,
            grounding TEXT,
            image_tag TEXT,
            date TEXT,
            model_response TEXT
        )
    """)
    conn.commit()
    conn.close()

create_db()



def store_experiment_result(image_name, tags, response):
    """Safe database insertion with all required fields"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO experiment_results_test2
        (image_name, question, context, object, grounding, image_tag, date, model_response)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        image_name,
        tags["question"],
        tags["context"],
        tags["object"],
        tags["grounding"],
        tags["image"],
        tags.get("date", "2023-01-01"),
        response
    ))

    conn.commit()
    conn.close()
    print(f"Stored result for {image_name}")

In [ ]:
import re
def clean_response(response):
    """Clean the response by removing unnecessary tags and repetitive text."""
    # Remove unwanted tags and noise
    unwanted_patterns = [
        r"<image>.*?</image>",  # Remove <image> tags and their content
        r"<grounding>.*?</grounding>",  # Remove <grounding> tags
        r"<context>.*?</context>",  # Remove <context> tags
        r"<image>.*?</image>",  # Remove <image> tags
        r"<object>.*?</object>",  # Remove <object> tags
        r"What can you see in this Sentinel-2A satellite image of.*?\?",  # Remove repetitive question
    ]

    for pattern in unwanted_patterns:
        response = re.sub(pattern, "", response, flags=re.DOTALL)

    # Remove extra spaces and newlines
    response = " ".join(response.split())

    return response.strip()


In [ ]:
#ran over 10 images in the drive
import os
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch
import pandas as pd


# Disable oneDNN optimizations if needed
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Set the path where your data is stored
datapath = r"/content"  # <-- Update this path to where your images are stored

# Load the Kosmos-2 model (this will download ~7GB on first run)
model_source = "microsoft/kosmos-2-patch14-224"
processor = AutoProcessor.from_pretrained(model_source)
model = AutoModelForVision2Seq.from_pretrained(model_source)

if torch.cuda.is_available():
    model = model.to("cuda")
    print("Using CUDA")

# ✅ Generate prompts for multiple images
# ✅ Generate prompts for multiple images
def generate_prompt(tags):

    """
    Generates a structured paratactic prompt.

    :param question: The main question.
    :param tags: Keyword arguments representing tag-value pairs.
    :return: Formatted prompt string.
    """
    prompt = f"""<grounding> {tags['grounding']} </grounding>
    <image> {tags['image']} </image>
    <context> {tags['context']} </context>
    <object> {tags['object']} </object>
    {tags['question']}"""
    return prompt

# ✅ List of input images to process
image_files = [
    "LA-GettyMuseum_openEO-2025-01-12Z_34_118_1738974467.png",
    "Damascus_openEO_2025-01-14Z_33_36_1738966392.png",
    "ndvi_ndvi_Paris_openEO_2025-02-01Z_anthropic.png",
    "Canberra_openEO_2025-01-13Z_35_149_1738976955.png",
    "fmr_fmr_openEO_2025-01-24Z_chatgpt_copper.png",
    "ndvi_ndvi_Paris_openEO_2025-02-01Z_chatgpt.png",
    "openEO_2024-12-30Z_36_111_1738893776.png",
    "openEO_2025-01-06Z_37_114_1738892548.png",
    "openEO_2024-12-31Z_34_118_1738890849.png",
    "openEO_2025-01-29Z_25_82_1738892257.png",
    "openEO_2025-01-30Z_51_0_1738890156.png",
    "openEO_2024-12-31Z_32_117_1738891932.png",
    "ndvi_ndvi_Paris_openEO_2025-02-01Z_anthropic.png"

]

def metadata(filename):
  parts = filename.split("_")
  if len(parts)>0:
    location = parts[0]
  else:
    location = ""

  for i in parts:
    if len(i) == 10 and i[4] == "-" and i[7] == "-":
      date = i
      break
    else:
      date = ""

  if "fmr" in filename.lower():
    additional_info = "Ferrous Mineral Ratio (FMR)"
  elif "ndvi" in filename.lower():
    additional_info = "Normalized Difference Vegetation Index (NDVI)"
  else:
    additional_info = "Unknown"

  return {"location": location, "date": date, "additional_info": additional_info}



for img_file in image_files:
    try:
        # Extract metadata
        meta_data = metadata(img_file)
        location = meta_data["location"]
        date = meta_data["date"]
        info = meta_data["additional_info"]

        # Generate question

        question = f"What can you see in this Sentinel-2A satellite image of {location}?"

        # Prepare tags
        tags = {
            "question": question,
            "grounding": f"Sentinel-2A satellite data.",
            "context": f"This is a satellite image derived from Sentinel-2A of {location} captured on {date}.",
            "image": f"Analyze the {info} in the image.",
            "object": f"Identify key objects in the image related to {info}.",
            "date": date
        }


        # Generate prompt
        prompt = generate_prompt(tags)


        # Process image
        image = Image.open(os.path.join(datapath, img_file))
        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding=True
        )

        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                min_length=8,
                num_beams=5,
                max_new_tokens=256,
                length_penalty=1.0,
                early_stopping = True
            )

        response = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        # Clean the response
        cleaned_response = clean_response(response)

        # Store results
        store_experiment_result(img_file, tags, cleaned_response)
        print(f"Processed {img_file}\nResponse: {cleaned_response}\n{'='*80}")

    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")

def export_to_csv():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM experiment_results_test2", conn)
    df.to_csv("experiment_results_test.csv", index=False)
    conn.close()
    print("Data exported to experiment_results_test2.csv")

export_to_csv()




In [ ]:
!pip install qdrant-client sentence-transformers




In [ ]:
#Using qdrant to vectorize promp
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

# Replace with your actual URL and API Key
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")
QDRANT_URL = os.environ.get("QDRANT_URL")

client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)


In [ ]:
# Disabled historical destructive collection reset.
raise RuntimeError("Use the maintained vector_store adapter instead")


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
documents = [
    {"id": 1, "image_name": "LA-GettyMuseum_openEO-2025-01-12Z_34_118_1738974467.png", "caption": "In the image, you can see several key objects related to the Unknown. These objects include a city, a mountain range, and a river. The city is located in the middle of the mountain range. The mountain range is located on the left side of the image and the river is on the right side. The presence of the city and mountain range suggests that the image was taken during a time when the city was located near the mountains. The river also indicates that the city is situated near a body of water, which could be a river, a lake, or a coastal area. It is important to note that the exact location of the unknown objects cannot be determined from the image alone. However, it can be inferred that they are located in a region with a significant amount of natural or man-made features, such as mountains, rivers, or coastal areas."},
    {"id": 2, "image_name": "Damascus_openEO_2025-01-14Z_33_36_1738966392.png", "caption": "In the image, you can see an aerial view of the city of Damascus, Syria. Damascus is located in the Middle East, and it is the capital city of the Syrian Arab Republic. The city has a population of more than 10 million people, making it one of the largest cities in the world. The image shows a densely populated urban area with numerous buildings, roads, and other infrastructure. There are numerous cars, trucks, and buses scattered throughout the city, indicating that it is a busy and bustling city. Some of the vehicles can be seen on the streets, while others are parked or parked near the buildings. Overall, the image depicts a bustling city with a mix of residential and commercial structures."},
]


In [ ]:
vectors = embedder.encode([doc["caption"] for doc in documents])

client.upsert(
    collection_name=collection_name,
    points=[
        {
            "id": doc["id"],
            "vector": vector.tolist(),
            "payload": doc  # Store image_name and caption
        }
        for doc, vector in zip(documents, vectors)
    ]
)


In [ ]:
import requests


DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {DEEPSEEK_API_KEY}"
}



Retrieving Contexts and building deepseek RAG function

In [ ]:
def retrieve_context(query, top_k=2):
    query_vector = embedder.encode(query).tolist()
    search_result = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=top_k
    )
    return [hit.payload['caption'] for hit in search_result]

# DeepSeek Query with RAG
def query_with_rag(prompt):
    # Step 1: Retrieve context from Qdrant
    contexts = retrieve_context(prompt)
    combined_context = "\n\n".join(contexts)

    # Step 2: Build augmented prompt
    augmented_prompt = f"""Based only on the following satellite image descriptions, answer the user's question.

    Context:
{combined_context}

Question:
{prompt}
"""

    # Step 3: Query DeepSeek
    return query_deepseek(augmented_prompt)

In [ ]:
import requests


DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {DEEPSEEK_API_KEY}"
}

#No RAG function
def query_deepseek(prompt):
  data = {
      "model": "deepseek-chat",
      "messages": [
          {"role": "system", "content": "You are an assistant that processes geosatellite imagery."},
          {"role": "user", "content": prompt}
      ],
      "temperature": 0.5
  }

  response = requests.post(
      "https://api.deepseek.com/v1/chat/completions",
      headers=headers,
      json=data
  )
  print(response.json())


  if response.ok:
    result = response.json()
    return result['choices'][0]['message']['content']
  else:
    raise Exception(f"Error {response.status_code}: {response.text}")


In [ ]:
def retrieve_context(query, top_k=2):
    query_vector = embedder.encode(query).tolist()
    search_result = client.search(
        collection_name=collection_name,
        query_vector=query_vector,
        limit=top_k
    )
    return [i.payload['caption'] for i in search_result]

#  DeepSeek Query with RAG
def query_with_rag(prompt):
    # Step 1: Retrieve context from Qdrant
    contexts = retrieve_context(prompt)
    combined_context = "\n\n".join(contexts)

    # Step 2: Build augmented prompt
    augmented_prompt = f"""Based only on the following satellite image descriptions, answer the user's question.

Context:
{combined_context}

Question:
{prompt}
"""

    return query_deepseek(augmented_prompt)


WITHOUT RAG FUNCTIONALITY

In [ ]:


prompt_no_rag = f"""

Context:
{captions}

Question:
What are the features of this image?
"""



# With RAG
rag_response = query_with_rag(prompt)
print("With RAG:", rag_response)

# Without RAG
no_rag_response = query_deepseek(prompt_no_rag)
print("Without RAG:", no_rag_response)
